# extracting patches

In [ ]:
! pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 14.3 MB/s eta 0:00:00


In [ ]:
import SimpleITK as sitk
import numpy as np
import os
import random

def load_images(t2_path, ciss_path):
    t2_img = sitk.ReadImage(t2_path)
    ciss_img = sitk.ReadImage(ciss_path)
    return t2_img, ciss_img

def compute_brain_mask(t2_img):
    otsu_filter = sitk.OtsuThresholdImageFilter()
    mask = otsu_filter.Execute(t2_img)
    mask = sitk.Cast(mask > 0, sitk.sitkUInt8)
    return mask

def is_valid_patch(mask, start_idx, size, existing_patches):
    patch_region = sitk.RegionOfInterest(mask, size, start_idx)
    patch_array = sitk.GetArrayFromImage(patch_region)
    brain_voxel_ratio = np.mean(patch_array > 0)

    if brain_voxel_ratio < 0.5:
        return False

    patch_volume = np.prod(size)
    for ep_start in existing_patches:
        overlap_min = [max(start_idx[d], ep_start[d]) for d in range(3)]
        overlap_max = [min(start_idx[d]+size[d], ep_start[d]+size[d]) for d in range(3)]
        overlap_size = [max(0, overlap_max[d]-overlap_min[d]) for d in range(3)]
        overlap_volume = np.prod(overlap_size)
        if overlap_volume / patch_volume > 0.2:
            return False
    return True

def extract_and_save_patches(t2_img, ciss_img, mask, out_dir, subj_id, team_id, patch_count=5, patch_size=(64,64,64)):
    np.random.seed(42)
    random.seed(42)
    size = patch_size
    shape = t2_img.GetSize()
    extracted_patches = []
    patches_coords = []
    attempts = 0

    while len(extracted_patches) < patch_count and attempts < 10000:
        # pick random center, then convert to start_idx
        center = [random.randint(size[d]//2, shape[d] - size[d]//2) for d in range(3)]
        start_idx = [c - size[d]//2 for d, c in enumerate(center)]

        if not is_valid_patch(mask, start_idx, size, patches_coords):
            attempts += 1
            continue

        # Extract patches for T2 and CISS
        t2_patch = sitk.RegionOfInterest(t2_img, size, start_idx)
        ciss_patch = sitk.RegionOfInterest(ciss_img, size, start_idx)

        patch_idx_str = f"{len(extracted_patches):02d}"
        t2_filename = f"subj{subj_id}_team{team_id}_patch{patch_idx_str}_t2_n4.nii.gz"
        ciss_filename = f"subj{subj_id}_team{team_id}_patch{patch_idx_str}_ciss.nii.gz"

        sitk.WriteImage(t2_patch, os.path.join(out_dir, t2_filename))
        sitk.WriteImage(ciss_patch, os.path.join(out_dir, ciss_filename))

        extracted_patches.append((t2_patch, ciss_patch))
        patches_coords.append(start_idx)
        attempts = 0  # reset attempts once success

    print(f"Extracted and saved {len(extracted_patches)} patches for subject {subj_id}, team {team_id}.")

# Example usage:
t2_path = "/content/drive/MyDrive/be5370_hw1/n4_corrected_subj_018.nii.gz"
ciss_path = "/content/drive/MyDrive/be5370_hw1/hemi_training/subj_018_ciss.nii.gz"
output_directory = "/content/drive/MyDrive/be5370_hw1/patch_output"
subject_id = "018"
team_id = "07"

os.makedirs(output_directory, exist_ok=True)

t2_img, ciss_img = load_images(t2_path, ciss_path)
mask = compute_brain_mask(t2_img)
extract_and_save_patches(t2_img, ciss_img, mask, output_directory, subject_id, team_id)


Extracted and saved 5 patches for subject 018, team 07.


## II-D

In [ ]:
import itertools, re, json
from pathlib import Path


# 1) Some basic functions

def load_seg(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    arr = np.squeeze(arr)
    return img, arr

def unique_labels(arr):
    """Return sorted non-background labels present in arr."""
    labs = np.unique(arr)
    labs = labs[labs != 0]
    return labs.tolist()

def dice_binary(A, B):
    """DSC for two boolean arrays. Handles empty sets -> returns np.nan."""
    inter = np.logical_and(A, B).sum()
    a = A.sum()
    b = B.sum()
    if a == 0 and b == 0:
        return np.nan  # undefined (both empty)
    return (2.0 * inter) / (a + b)

def dice_for_label(arr1, arr2, label):
    return dice_binary(arr1 == label, arr2 == label)

def parse_meta_from_name(fname):
    """
    Parse subject, team, patch, member id based on file name
    Example:
      subj018_team007_patch001_L_seg.nii.gz
      subj018_team07_patch01_memberWP_seg.nii.gz
    Missing fields are set to None.
    """
    name = Path(fname).name

    def g(pat, s):
        m = re.search(pat, s, flags=re.IGNORECASE)
        return m.group(1) if m else None

    subject = g(r"subj[_-]?(\d+)", name)
    team    = g(r"team[_-]?0*(\d+)", name)
    patch   = g(r"patch[_-]?0*(\d+)", name)
    member  = g(r"member([A-Za-z0-9]+)", name)
    if member is None:
        member = g(r"[_-]([A-Za-z]{2,})[_-]?seg", name)

    return dict(subject=subject, team=team, patch=patch, member=member, filename=name)

# 2) Load the data

# Set to the folder holding all *_seg.nii.gz files
ROOT = Path("/content/drive/MyDrive/be5370_hw1/patch_seg_2c")

files = sorted(list(ROOT.rglob("*_seg.nii.gz")))
assert files, "No segmentation files found."

records = []
for f in files:
    meta = parse_meta_from_name(f.name)
    img, arr = load_seg(f)
    rec = {
        **meta,
        "path": str(f),
        "spacing": img.GetSpacing(),
        "origin": img.GetOrigin(),
        "direction": img.GetDirection(),
        "labels": json.dumps(unique_labels(arr)),  # for inspection
    }
    records.append(rec)

df_files = pd.DataFrame(records)
display(df_files)

,subject,team,patch,member,filename,path,spacing,origin,direction,labels
0,018,7,1,kjn,subj018_team007_patch001_kjn_seg.nii.gz,/content/drive/MyDrive/be5370_hw1/patch_seg_2c...,"(0.30000001192092896, 0.30000001192092896, 0.3...","(10.151820182800293, 1.347430944442749, -90.10...","(-0.0, -1.0, 0.0, 0.0, -0.0, -1.0, 1.0, 0.0, 0.0)","[1, 2]"
1,018,7,1,L,subj018_team007_patch001_memberL_seg.nii.gz,/content/drive/MyDrive/be5370_hw1/patch_seg_2c...,"(0.30000001192092896, 0.30000001192092896, 0.3...","(10.151820182800293, 1.347430944442749, -90.10...","(-0.0, -1.0, 0.0, 0.0, -0.0, -1.0, 1.0, 0.0, 0.0)","[1, 2]"
2,018,7,1,WP,subj018_team07_patch01_memberWP_seg.nii.gz,/content/drive/MyDrive/be5370_hw1/patch_seg_2c...,"(0.30000001192092896, 0.30000001192092896, 0.3...","(10.151820182800293, 1.347430944442749, -90.10...","(-0.0, -1.0, 0.0, 0.0, -0.0, -1.0, 1.0, 0.0, 0.0)","[1, 2]"
3,021,7,4,kjn,subj021_team007_patch004_kjn_seg.nii.gz,/content/drive/MyDrive/be5370_hw1/patch_seg_2c...,"(0.30000001192092896, 0.30000001192092896, 0.3...","(81.89999389648438, -52.5496826171875, -139.10...","(-0.0, 1.0, 0.0, 0.0, -0.0, -1.0, -1.0, 0.0, 0.0)","[1, 2]"
4,021,7,4,L,subj021_team07_patch004_memberL_seg.nii.gz,/content/drive/MyDrive/be5370_hw1/patch_seg_2c...,"(0.30000001192092896, 0.30000001192092896, 0.3...","(81.89999389648438, -52.5496826171875, -139.10...","(-0.0, 1.0, 0.0, 0.0, -0.0, -1.0, -1.0, 0.0, 0.0)","[1, 2]"
5,021,7,4,WP,subj021_team07_patch04_memberWP_seg.nii.gz,/content/drive/MyDrive/be5370_hw1/patch_seg_2c...,"(0.30000001192092896, 0.30000001192092896, 0.3...","(81.89999389648438, -52.5496826171875, -139.10...","(-0.0, 1.0, 0.0, 0.0, -0.0, -1.0, -1.0, 0.0, 0.0)","[1, 2]"


In [ ]:

# Save STAPLE consensus images

OUTPUT_DIR = Path("/content/drive/MyDrive/be5370_hw1/be5370_hw1/patch_seg_2c")
OUTPUT_DIR.mkdir(exist_ok=True)

for gkey, gdf in df_files.groupby("group_key"):
    subj = gdf.iloc[0]["subject"]
    patch = gdf.iloc[0]["patch"]

    # Build consensus
    cons_img = staple_consensus_multilabel(gdf["path"].tolist())

    # Construct output filename (similar style as your member segmentations)
    out_path = OUTPUT_DIR / f"subj{subj}_team07_patch{patch}_final.nii.gz"

    # Save consensus segmentation
    sitk.WriteImage(cons_img, str(out_path))
    print(f"Saved STAPLE consensus: {out_path}")
